# Policy Visualization Experiment
This notebook visualizes the behavior of previously trained policies on Adroit manipulation tasks.
You can switch between offline, online, or finetuned models, and optionally save rendered videos to file.

In [ ]:
import d3rlpy
import minari
import time
import imageio
import imageio.v3 as iio
import numpy as np
import os 

## Experiment Setup
Select the type of experiment, task, algorithm, and number of episodes to render.

In [ ]:
# Select experiment, tasks and algorithm to visualize

experiment = 'offline'   # 'offline', 'finetuning', or 'online'
tasks = ['hammer']       # ['relocate', 'door', 'pen', 'hammer']
algorithm = 'iql'        # 'iql', 'cql', 'bc', 'td3bc', 'awac'

repeat = 10              # Number of rollouts to perform for each task

save_mp4 = False         # Whether to save visualizations as .mp4 files

## Visualization of policies
This function executes and renders a policy for a fixed number of episodes.

In [ ]:
def visualize(env, policy):
    try:
        for i in range(repeat):
            obs, _ = env.reset()
            done = False
            total_reward = 0
            frames = []
            n = 0

            # Run a rollout using the given policy
            while not done:
                action = policy.predict(obs[None])[0]
                obs, reward, terminated, truncated, _ = env.step(action)

                done = terminated or n > 500 or truncated
                total_reward += reward
                n += 1

                # Save frame for video rendering or render in real-time
                if save_mp4:
                    frame = env.render()
                    frames.append(frame)
                else:
                    time.sleep(0.01)

            # Create directory and save video
            folder_path = "videos"
            os.makedirs(folder_path, exist_ok=True)

            if save_mp4:
                mp4_path = f"{folder_path}/{task}_{experiment}_{algorithm}.mp4"
                iio.imwrite(mp4_path, np.array(frames), fps=30, codec='libx264')
                print(f"MP4 saved to: {mp4_path}")

            print(f"Episode {i+1} finished with return: {total_reward:.2f}")

    except Exception as e:
        print(e)

    env.close()


## Policy execution

This section loads the saved policy, recovers the corresponding environment, and runs the visualization function for a number of episodes.

In [ ]:
# Choose the appropriate rendering mode based on whether a video will be saved
if save_mp4:
    render_mode = "rgb_array"  # Required for saving frames as video
else:
    render_mode = "human"      # Render directly to screen

for task in tasks:
    # Load the trained policy from file
    policy = d3rlpy.load_learnable(f"policies/{experiment}/{task}_{algorithm}_policy.d3")

    # Load the environment from Minari and recover it with proper rendering
    environment = minari.load_dataset(f"D4RL/{task}/human-v2").recover_environment(render_mode=render_mode)

    # Visualize the policy in the selected environment
    visualize(environment, policy)
